# GraphTrust large run: saas_scaleup (Kaggle adaptation, non-Colab)

**This notebook is a documented deviation from `docs/COLAB_RUNBOOK.md`.** The
original large-profile runbook requires an authenticated Google Colab
CPU high-RAM runtime (>=20 GiB RAM) and persistence via Google Drive. Colab Pro
access (required for a High-RAM runtime) was not available to the researcher
running this evidence, so this notebook runs the same frozen
`configs/large.yaml` workload on Kaggle Notebooks instead.

**Equivalence to the Colab-specified environment has not been independently
verified.** The receipt this notebook produces truthfully records
`is_google_colab: false` and adds `is_kaggle: true` and `platform_note` fields
so this cannot be mistaken for Colab-sourced evidence. Any resulting figures,
tables, or claims must cite this notebook and this deviation explicitly, not
be pooled silently into Colab-labeled large-profile results.

Run every cell from top to bottom in a Kaggle Notebook with **Internet: on**
and, ideally, a session with the most available RAM your Kaggle tier offers
(GPU-enabled sessions on Kaggle currently offer the most system RAM even
though this workload does not use the GPU itself; verify current limits at
kaggle.com/code before starting since Kaggle's free-tier resources change
over time).


## 0. Settings and persistent workspace (Kaggle Datasets, not Drive)

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

PROFILE = "saas_scaleup"
SEED = 2750159
REPO = "https://github.com/sronters/graph.git"
BRANCH = "main"  # PR #2 must be merged before this run.
ROOT = Path("/kaggle/working/graphtrust")

# Kaggle has no Google Drive mount. /kaggle/working persists for the life of
# the session and its contents are downloadable from the notebook's Output
# tab. For persistence across sessions (Colab's Drive-resume behavior),
# manually save /kaggle/working/GraphTrustLarge as a Kaggle Dataset via
# "Save Version" -> "Save & Run All" and attach it as an input on your next
# session if you need to resume a run that a session limit interrupted.
PERSIST = ROOT.parent / "GraphTrustLarge" / PROFILE / str(SEED)
DATA_ROOT = PERSIST / "data"
OUTPUT_ROOT = PERSIST / "artifacts"
PERSIST.mkdir(parents=True, exist_ok=True)
print({"profile": PROFILE, "seed": SEED, "persistent_workspace": str(PERSIST)})


## 1. Clone the final source and install the locked environment

In [ ]:
if not (ROOT / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(ROOT)], check=True
    )
os.chdir(ROOT)
subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", "--detach", f"origin/{BRANCH}"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync", "--frozen", "--all-extras"], check=True)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print({"git_commit": COMMIT, "python": sys.version, "platform": platform.platform()})


## 2. Fail-fast capacity gate and smoke test (Kaggle environment marker)

In [ ]:
import psutil

available_ram_gib = psutil.virtual_memory().available / 2**30
drive_free_gib = shutil.disk_usage(PERSIST).free / 2**30

# Kaggle sets KAGGLE_KERNEL_RUN_TYPE for every notebook session. This is the
# honest Kaggle-equivalent of the original notebook's COLAB_RELEASE_TAG check
# -- it confirms you are actually inside a Kaggle Notebook, not a spoof of
# the Colab check.
assert os.getenv("KAGGLE_KERNEL_RUN_TYPE"), "This must run inside a Kaggle Notebook."
assert available_ram_gib >= 20, (
    f"This session has only {available_ram_gib:.1f} GiB available. Try a "
    "GPU-enabled Kaggle session (more system RAM is allocated even though "
    "the GPU itself is unused), or check current Kaggle tier limits."
)
assert drive_free_gib >= 15, (
    f"Free at least 15 GiB of working storage; only {drive_free_gib:.1f} GiB is available."
)
subprocess.run(
    [
        "uv", "run", "graphtrust", "generate",
        "--profile", PROFILE, "--scale", "large", "--seed", str(SEED),
        "--variants", "injected_mixed", "--dry-run",
    ],
    check=True,
)
subprocess.run(
    ["uv", "run", "pytest", "-q", "tests/integration/test_experiment_runner.py"], check=True
)
print(
    {
        "available_ram_gib": round(available_ram_gib, 1),
        "drive_free_gib": round(drive_free_gib, 1),
        "smoke_test": "passed",
    }
)


## 3. Generate or reuse the deterministic 2.5-million-edge graph

In [ ]:
dataset = DATA_ROOT / PROFILE / "large" / str(SEED) / "injected_mixed"
if not (dataset / "checksums.sha256").exists():
    subprocess.run(
        [
            "uv", "run", "graphtrust", "generate",
            "--profile", PROFILE, "--scale", "large", "--seed", str(SEED),
            "--variants", "injected_mixed", "--output-root", str(DATA_ROOT),
        ],
        check=True,
    )
subprocess.run(["uv", "run", "graphtrust", "validate-data", "--dataset", str(dataset)], check=True)
print({"dataset": str(dataset), "status": "checksum-verified"})


## 4. Run or resume the immutable bounded analysis

In [ ]:
subprocess.run(
    [
        "uv", "run", "python", "scripts/run_large_profiles.py",
        "--profile", PROFILE, "--seed", str(SEED),
        "--data-root", str(DATA_ROOT), "--config", "configs/large.yaml",
        "--output", str(OUTPUT_ROOT),
    ],
    check=True,
)
receipt = OUTPUT_ROOT / "large_run_receipt.json"
receipt_data = json.loads(receipt.read_text())

# Honest check for this deviation path: this run is expected to be NOT
# Colab. is_google_colab is computed upstream as "COLAB_RELEASE_TAG" in
# os.environ, so on Kaggle it truthfully evaluates to False. We assert that
# explicitly here (rather than asserting True, as the original Colab
# notebook does) so this receipt can never be silently confused with a real
# Colab receipt downstream.
assert receipt_data["is_google_colab"] is False, (
    "Expected is_google_colab: false on Kaggle. If this is True, something "
    "is wrong with the execution environment detection -- stop and check "
    "before trusting this receipt."
)
assert len(receipt_data["profiles"]) == 1
assert all(item["verified"] for item in receipt_data["profiles"])

# Explicit, additive platform markers so downstream tooling and readers can
# never mistake this for Colab-sourced evidence.
receipt_data["is_kaggle"] = bool(os.getenv("KAGGLE_KERNEL_RUN_TYPE"))
receipt_data["kaggle_kernel_run_type"] = os.getenv("KAGGLE_KERNEL_RUN_TYPE")
receipt_data["platform_note"] = (
    "Generated on Kaggle Notebooks as a documented deviation from "
    "docs/COLAB_RUNBOOK.md (Colab Pro/High-RAM was unavailable). "
    "Equivalence to the Colab-specified environment has not been "
    "independently verified."
)
receipt.write_text(json.dumps(receipt_data, indent=2, sort_keys=True) + "\n")
print(json.dumps(receipt_data, indent=2))


## 5. Build the verified evidence ZIP (Kaggle Output tab, not Drive download)

In [ ]:
export_root = PERSIST / "export"
export_root.mkdir(parents=True, exist_ok=True)
shutil.copy2(dataset / "manifest.json", export_root / "dataset_manifest.json")
shutil.copy2(dataset / "checksums.sha256", export_root / "dataset_checksums.sha256")
shutil.copy2(OUTPUT_ROOT / "large_run_receipt.json", export_root / "large_run_receipt.json")
archive_base = PERSIST / f"GraphTrust_large_{PROFILE}_{SEED}_kaggle"
archive = Path(shutil.make_archive(str(archive_base), "zip", PERSIST, "artifacts"))
archive_digest = hashlib.sha256()
with archive.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        archive_digest.update(chunk)
archive_sha256 = archive_digest.hexdigest()
download_receipt = {
    "archive": archive.name,
    "archive_sha256": archive_sha256,
    "git_commit": COMMIT,
    "profile": PROFILE,
    "seed": SEED,
    "platform": "kaggle",
    "platform_note": (
        "Documented deviation from docs/COLAB_RUNBOOK.md. Colab Pro/High-RAM "
        "was unavailable; this evidence was generated on Kaggle Notebooks "
        "instead. Not verified equivalent to the Colab-specified environment."
    ),
}
download_receipt_path = export_root / f"GraphTrust_large_{PROFILE}_{SEED}_kaggle_download_receipt.json"
download_receipt_path.write_text(json.dumps(download_receipt, indent=2, sort_keys=True) + "\n")
print(download_receipt)

# Kaggle has no files.download() equivalent to Colab's. Everything under
# /kaggle/working is downloadable from the notebook's "Output" tab in the
# Kaggle UI after the session finishes (or via "Save Version" -> a Data
# output). Copy the two files you need to send back into /kaggle/working
# explicitly so they show up there without digging through subfolders.
final_outputs = Path("/kaggle/working") / "final_outputs"
final_outputs.mkdir(parents=True, exist_ok=True)
shutil.copy2(archive, final_outputs / archive.name)
shutil.copy2(download_receipt_path, final_outputs / download_receipt_path.name)
print({"download_from_kaggle_output_tab": str(final_outputs)})


## What to send back

From the Kaggle notebook's **Output** tab (right sidebar after the run
finishes, or via "Save Version"), download everything under
`final_outputs/`:

- `GraphTrust_large_<profile>_<seed>_kaggle.zip`
- `GraphTrust_large_<profile>_<seed>_kaggle_download_receipt.json`

Do not report runtime or memory numbers, and do not present this as Colab
evidence, unless you have re-read `platform_note` in both the run receipt and
the download receipt and included that context anywhere these numbers are
cited.